# Feature Engineering

## 1. Setup and Imports

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(sys.path[0])

c:\Users\rebec\Documents\Thesis


In [2]:
import numpy as np
import pandas as pd


from src.config import (
    TARGET,
    MIGRAINE_FILTER,
    SEVERITY_ORDER,
    PHYSIOLOGICAL_BINARY_FEATURES,
    PHYSIOLOGICAL_ORDINAL_FEATURES,
    ENVIRONMENTAL_BINARY_FEATURES,
    LIFESTYLE_BINARY_FEATURES,
    TEMPORAL_FEATURES,
    WEATHER_RAW_FEATURES,
    WEATHER_CHANGE_24H_FEATURES,
    WEATHER_CHANGE_48H_FEATURES,
    WEATHER_CHANGE_72H_FEATURES,
    WEATHER_ENRICHED_DATA_FILE,
    GROUP,
    FEATURE_ENGINEERED_DATA_FILE,
    WEATHER_CYCLICAL_FEATURES,
    PHYSIOLOGICAL_LIFESTYLE_FEATURES,
    ENVIRONMENTAL_TEMPORAL_FEATURES,
    COMBINED_FEATURES,
    FEATURE_SETS,
)

## 2. Load Modeling Data

In [3]:
migraine_data = pd.read_csv(WEATHER_ENRICHED_DATA_FILE)

print(f"Dataset shape: {migraine_data.shape}")
print(f"Dataset columns: {migraine_data.columns.tolist()}")

Dataset shape: (302, 66)
Dataset columns: ['No.', 'Registration No.', 'Preventive Medication Use', 'Start Time', 'Index (VAS Style Score)', 'Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis', 'Internal Factor: Stress', 'Internal Factor: Oversleeping', 'Internal Factor: Sleep Deprivation', 'Internal Factor: Exercise', 'Internal Factor: Lack of Exercise', 'Internal Factor: Physical Fatigue', 'Internal Factor: Menstrual Cycle: Menstruation', 'Internal Factor: Menstrual Cycle: Ovulation', 'Internal Factor: Excessive Emotional Change', 'External Factor: Excessive Sunlight', 'External Factor: Noise', 'External Factor: Inappropriate Lighting', 'External Factor: Specific Smell (cosmetics, perfume, etc.)', 'Other Factor: Excessive Alcohol', 'Other Factor: Irregular Meals (fasting, etc.)', 'Other Factor: Overeating', 'Other Factor: Excessive Caffeine', 'Other Factor: Excessive Smoking', 'Other Factor: Cheese/Chocolate', 'Other Factor: Travel', 'Other Factor: Other', 'Vigorous Exercise', 'M

In [4]:
migraine_data["Start Time"] = pd.to_datetime(
    migraine_data["Start Time"],
    errors = "coerce"
)

migraine_data["Weather_Hour"] = pd.to_datetime(
    migraine_data["Weather_Hour"],
    errors = "coerce"
)

migraine_data[["Start Time", "Weather_Hour"]]

,Start Time,Weather_Hour
0,2014-12-21 10:18:00,2014-12-21 10:00:00
1,2014-09-29 11:34:00,2014-09-29 12:00:00
2,2014-11-29 11:20:00,2014-11-29 11:00:00
3,2014-11-24 20:50:00,2014-11-24 21:00:00
4,2015-01-25 08:00:00,2015-01-25 08:00:00
...,...,...
297,2014-10-28 09:00:00,2014-10-28 09:00:00
298,2014-10-03 06:31:00,2014-10-03 07:00:00
299,2015-01-11 08:00:00,2015-01-11 08:00:00
300,2014-10-08 03:10:00,2014-10-08 03:00:00


## 3. Define Target and Grouping Variable

In [5]:
migraine_data[TARGET].value_counts(dropna=False)

Severity_VAS
Moderate    138
Severe      109
Mild         55
Name: count, dtype: int64

In [6]:
migraine_data[TARGET].dtype

<StringDtype(storage='python', na_value=nan)>

In [7]:
migraine_data[TARGET] = pd.Categorical(
    migraine_data[TARGET],
    categories=SEVERITY_ORDER,
    ordered=True
)

print(migraine_data[TARGET].dtype)
print(migraine_data[TARGET].value_counts().sort_index())

category
Severity_VAS
Mild         55
Moderate    138
Severe      109
Name: count, dtype: int64


In [8]:
GROUP = "Registration No."

print(f"Number of migraine episodes: {len(migraine_data)}")
print(f"Number of participants: {migraine_data[GROUP].nunique()}")
print(f"Missing participant IDs: {migraine_data[GROUP].isna().sum()}")

Number of migraine episodes: 302
Number of participants: 48
Missing participant IDs: 0


In [9]:
feature_data = migraine_data.copy()

print(f"Original data shape: {migraine_data.shape}")
print(f"Feature data shape: {feature_data.shape}")

print("Dataframes identical:", migraine_data.equals(feature_data))

Original data shape: (302, 66)
Feature data shape: (302, 66)
Dataframes identical: True


## 4. Define Candidate Feature Sets

In [10]:
feature_groups = {
    "Physiological Binary": PHYSIOLOGICAL_BINARY_FEATURES,
    "Physiological Ordinal": PHYSIOLOGICAL_ORDINAL_FEATURES,
    "Environmental Binary": ENVIRONMENTAL_BINARY_FEATURES,
    "Lifestyle Binary": LIFESTYLE_BINARY_FEATURES,
    "Temporal": TEMPORAL_FEATURES,
    "Weather Raw": WEATHER_RAW_FEATURES,
    "Weather Change 24h": WEATHER_CHANGE_24H_FEATURES,
    "Weather Change 48h": WEATHER_CHANGE_48H_FEATURES,
    "Weather Change 72h": WEATHER_CHANGE_72H_FEATURES
}

for group_name, features in feature_groups.items():
    print(f"\n{group_name}: {len(features)} features")
    print("-" * 50)
    
    for feature in features:
        print(feature)


Physiological Binary: 11 features
--------------------------------------------------
Preventive Medication Use
Internal Factor: Stress
Internal Factor: Oversleeping
Internal Factor: Sleep Deprivation
Internal Factor: Exercise
Internal Factor: Lack of Exercise
Internal Factor: Physical Fatigue
Internal Factor: Menstrual Cycle: Menstruation
Internal Factor: Menstrual Cycle: Ovulation
Internal Factor: Excessive Emotional Change
Other Factor: Irregular Meals (fasting, etc.)

Physiological Ordinal: 1 features
--------------------------------------------------
Total Exercise

Environmental Binary: 4 features
--------------------------------------------------
External Factor: Excessive Sunlight
External Factor: Noise
External Factor: Inappropriate Lighting
External Factor: Specific Smell (cosmetics, perfume, etc.)

Lifestyle Binary: 7 features
--------------------------------------------------
Other Factor: Excessive Alcohol
Other Factor: Overeating
Other Factor: Excessive Caffeine
Other Fac

## 5. Remove Redundant Features
Surface pressure and its 24-, 48-, and 72-hour change variables were excluded
from the candidate feature sets because exploratory analysis showed that they
were highly correlated with mean sea-level pressure (`pressure_msl`) and its
corresponding change variables.

Mean sea-level pressure was retained to avoid including redundant pressure
measurements in the predictive models.

## 6. Engineer Weather Features

### 6.1 Wind Direction
Wind direction is a circular variable measured from 0° to 360°. Because values near 0° and 360° represent similar directions, wind direction should not be treated as a standard continuous variable. The distribution and missingness of wind direction are first evaluated before applying a cyclical transformation.

In [11]:
feature_data["wind_direction_10m"].describe()

count    302.000000
mean     195.745033
std      103.952765
min        5.000000
25%       96.000000
50%      200.500000
75%      300.000000
max      360.000000
Name: wind_direction_10m, dtype: float64

In [12]:
print(
    f"Missing wind directions values: "
    f"{feature_data['wind_direction_10m'].isna().sum()}"
)

print(
    f"Minimum wind direction: "
    f"{feature_data['wind_direction_10m'].min()}"
)

print(
    f"Maximum wind direction: "
    f"{feature_data['wind_direction_10m'].max()}"
)

Missing wind directions values: 0
Minimum wind direction: 5
Maximum wind direction: 360


In [13]:
# Converting degrees to radians for wind direction
feature_data["wind_direction_sin"] = np.sin(np.deg2rad(feature_data["wind_direction_10m"]))
feature_data["wind_direction_cos"] = np.cos(np.deg2rad(feature_data["wind_direction_10m"]))

feature_data[
    ["wind_direction_10m", 
     "wind_direction_sin", 
     "wind_direction_cos"]
].head(10)

# feature_data[
#     ["wind_direction_sin", "wind_direction_cos"]
# ].describe()

,wind_direction_10m,wind_direction_sin,wind_direction_cos
0,321,-0.629320,0.777146
1,130,0.766044,-0.642788
2,333,-0.453990,0.891007
3,16,0.275637,0.961262
4,73,0.956305,0.292372
5,289,-0.945519,0.325568
6,346,-0.241922,0.970296
7,41,0.656059,0.754710
8,301,-0.857167,0.515038
9,248,-0.927184,-0.374607


In [14]:
feature_data[
    ["wind_direction_10m_change_24h", 
     "wind_direction_10m_change_48h", 
     "wind_direction_10m_change_72h"]
].describe()

,wind_direction_10m_change_24h,wind_direction_10m_change_48h,wind_direction_10m_change_72h
count,301.000000,300.000000,300.000000
mean,1.431894,-4.283333,-11.326667
std,87.474794,100.235991,100.639626
min,-180.000000,-180.000000,-179.000000
25%,-48.000000,-84.250000,-98.250000
50%,0.000000,-10.000000,-10.500000
75%,52.000000,75.000000,59.000000
max,179.000000,179.000000,179.000000


### 6.2 Weather Change Magnitude

Absolute weather-change features were created to represent the magnitude of change regardless of direction. The original signed change variables were retained to preserve information about whether each weather variable increased or decreased.

In [15]:
# Create weather change features
weather_change_variables = [
    "temperature",
    "relative_humidity",
    "pressure_msl",
    "wind_speed_10m"
]

for hours in [24, 48, 72]:
    for variable in weather_change_variables:
        source_feature = f"{variable}_change_{hours}h"
        new_feature = f"abs_{variable}_change_{hours}h"

        feature_data[new_feature] = feature_data[source_feature].abs()

In [16]:
# Identify and list all absolute weather change features
absolute_weather_features = [
    column
    for column in feature_data.columns
    if column.startswith("abs_")
]

print(f"Number of absolute weather features: {len(absolute_weather_features)}")

for feature in absolute_weather_features:
    print(feature)

Number of absolute weather features: 12
abs_temperature_change_24h
abs_relative_humidity_change_24h
abs_pressure_msl_change_24h
abs_wind_speed_10m_change_24h
abs_temperature_change_48h
abs_relative_humidity_change_48h
abs_pressure_msl_change_48h
abs_wind_speed_10m_change_48h
abs_temperature_change_72h
abs_relative_humidity_change_72h
abs_pressure_msl_change_72h
abs_wind_speed_10m_change_72h


### 6.3 Review Engineered Weather Features

The engineered weather features were reviewed for missing values and valid ranges before inclusion in the final candidate feature sets.

In [17]:
feature_data[absolute_weather_features].isna().sum()

abs_temperature_change_24h          1
abs_relative_humidity_change_24h    1
abs_pressure_msl_change_24h         1
abs_wind_speed_10m_change_24h       1
abs_temperature_change_48h          2
abs_relative_humidity_change_48h    2
abs_pressure_msl_change_48h         2
abs_wind_speed_10m_change_48h       2
abs_temperature_change_72h          2
abs_relative_humidity_change_72h    2
abs_pressure_msl_change_72h         2
abs_wind_speed_10m_change_72h       2
dtype: int64

In [18]:
feature_data[["wind_direction_sin", "wind_direction_cos"]].isna().sum()

wind_direction_sin    0
wind_direction_cos    0
dtype: int64

In [19]:
weather_change_features = (
    WEATHER_CHANGE_24H_FEATURES
    + WEATHER_CHANGE_48H_FEATURES
    + WEATHER_CHANGE_72H_FEATURES
)

feature_data[weather_change_features].isna().sum()

relative_humidity_change_24h     1
temperature_change_24h           1
pressure_msl_change_24h          1
wind_speed_10m_change_24h        1
wind_direction_10m_change_24h    1
relative_humidity_change_48h     2
temperature_change_48h           2
pressure_msl_change_48h          2
wind_speed_10m_change_48h        2
wind_direction_10m_change_48h    2
relative_humidity_change_72h     2
temperature_change_72h           2
pressure_msl_change_72h          2
wind_speed_10m_change_72h        2
wind_direction_10m_change_72h    2
dtype: int64

## 7. Engineer Temporal Features

### 7.1 Start Hour

Start hour is a cyclical variable because the end of one day is adjacent to the beginning of the next. Sine and cosine transformations are used to preserve this cyclical relationship.

In [20]:
feature_data["Start_Hour"].describe()

count    302.000000
mean       9.784768
std        5.243255
min        0.000000
25%        6.250000
50%        9.500000
75%       13.000000
max       23.000000
Name: Start_Hour, dtype: float64

In [21]:
print(
    f"Missing Start_Hour values: {feature_data['Start_Hour'].isna().sum()}\n",
    f"Minimum Start_Hour value: {feature_data['Start_Hour'].min()}\n",
    f"Maximum Start_Hour value: {feature_data['Start_Hour'].max()}"
)

Missing Start_Hour values: 0
 Minimum Start_Hour value: 0.0
 Maximum Start_Hour value: 23.0


### 7.2 Create cyclical Start Hour features
Convert migraine onset hour to cyclical features so that adjacent times across midnight are represented as close together

In [22]:
feature_data["Start_Hour_sin"] = np.sin(2 * np.pi * feature_data["Start_Hour"] / 24)
feature_data["Start_Hour_cos"] = np.cos(2 * np.pi * feature_data["Start_Hour"] / 24)

feature_data[
    ["Start_Hour",
     "Start_Hour_sin",
     "Start_Hour_cos"]
].head()

,Start_Hour,Start_Hour_sin,Start_Hour_cos
0,10.0,0.500000,-0.866025
1,11.0,0.258819,-0.965926
2,11.0,0.258819,-0.965926
3,20.0,-0.866025,0.500000
4,8.0,0.866025,-0.500000


In [23]:
feature_data[
    ["Start_Hour",
     "Start_Hour_sin",
     "Start_Hour_cos"]
].describe()

,Start_Hour,Start_Hour_sin,Start_Hour_cos
count,302.000000,302.000000,3.020000e+02
mean,9.784768,0.232066,-3.014902e-01
std,5.243255,0.667218,6.425814e-01
min,0.000000,-1.000000,-1.000000e+00
25%,6.250000,-0.258819,-8.660254e-01
50%,9.500000,0.500000,-5.000000e-01
75%,13.000000,0.866025,6.123234e-17
max,23.000000,1.000000,1.000000e+00


### 7.3 Review Time of Day

In [24]:
feature_data["Time_of_Day"].value_counts(dropna=False)

Time_of_Day
Morning      156
Afternoon     70
Night         50
Evening       26
Name: count, dtype: int64

In [25]:
feature_data["Time_of_Day"].dtype

<StringDtype(storage='python', na_value=nan)>

## 8. Review Engineered Features
The newly engineered features were reviewed to confirm that they were created successfully and contain the expected values before defining the final predictor sets.

In [26]:
engineered_features = [
    "wind_direction_sin",
    "wind_direction_cos",
    "Start_Hour_sin",
    "Start_Hour_cos"
] + absolute_weather_features

print(f"Number of engineered features: {len(engineered_features)}")

for feature in engineered_features:
    print(feature)

Number of engineered features: 16
wind_direction_sin
wind_direction_cos
Start_Hour_sin
Start_Hour_cos
abs_temperature_change_24h
abs_relative_humidity_change_24h
abs_pressure_msl_change_24h
abs_wind_speed_10m_change_24h
abs_temperature_change_48h
abs_relative_humidity_change_48h
abs_pressure_msl_change_48h
abs_wind_speed_10m_change_48h
abs_temperature_change_72h
abs_relative_humidity_change_72h
abs_pressure_msl_change_72h
abs_wind_speed_10m_change_72h


In [27]:
feature_data[engineered_features].isna().sum()

wind_direction_sin                  0
wind_direction_cos                  0
Start_Hour_sin                      0
Start_Hour_cos                      0
abs_temperature_change_24h          1
abs_relative_humidity_change_24h    1
abs_pressure_msl_change_24h         1
abs_wind_speed_10m_change_24h       1
abs_temperature_change_48h          2
abs_relative_humidity_change_48h    2
abs_pressure_msl_change_48h         2
abs_wind_speed_10m_change_48h       2
abs_temperature_change_72h          2
abs_relative_humidity_change_72h    2
abs_pressure_msl_change_72h         2
abs_wind_speed_10m_change_72h       2
dtype: int64

## 9. Define Final Feature Sets

Final predictor sets are defined in `config.py` to maintain consistent feature definitions across feature engineering and model development.

Three feature sets will be evaluated:

1. **Physiological + Lifestyle**
2. **Environmental + Temporal**
3. **Combined**

The following checks confirm that all predictors defined in each feature set are present in the feature-engineered dataset.

In [28]:
for set_name, features in FEATURE_SETS.items():
    print(f"{set_name}: {len(features)} features")

Physiological + Lifestyle: 19 features
Environmental + Temporal: 40 features
Combined: 59 features


In [30]:
for set_name, features in FEATURE_SETS.items():

    missing_features = [
        feature
        for feature in features
        if feature not in feature_data.columns
    ]

    print(f"\n{set_name}")
    print("-" * 50)

    if len(missing_features) == 0:
        print("All features are present.")
    else:
        print("Missing features:")      
        for feature in missing_features:
            print(f" - {feature}")


Physiological + Lifestyle
--------------------------------------------------
All features are present.

Environmental + Temporal
--------------------------------------------------
All features are present.

Combined
--------------------------------------------------
All features are present.


In [31]:
for set_name, features in FEATURE_SETS.items():

    duplicate_features = [
        feature
        for feature in features
        if features.count(feature) > 1
    ]

    print(f"\n{set_name}")
    print("-" * 50)

    if duplicate_features:
        print(duplicate_features)


Physiological + Lifestyle
--------------------------------------------------

Environmental + Temporal
--------------------------------------------------

Combined
--------------------------------------------------


In [32]:
excluded_predictors = [
    TARGET,
    GROUP,
    MIGRAINE_FILTER,
    "VAS",
    "Start Time",
    "Weather_Hour",
    "Start_Hour",
    "wind_direction_10m",
    "surface_pressure",
    "surface_pressure_change_24h",
    "surface_pressure_change_48h",
    "surface_pressure_change_72h",
]

unexpected_predictors = [
    feature
    for feature in excluded_predictors
    if feature in COMBINED_FEATURES
]

print("Excluded variables found in predictor set:")
print(unexpected_predictors)

Excluded variables found in predictor set:
[]


## 10. Save Feature-Engineered Dataset

In [33]:
modeling_columns = [
    GROUP,
    TARGET,
] + COMBINED_FEATURES

final_feature_data = feature_data[
    modeling_columns
].copy()

In [34]:
print(f"Rows: {final_feature_data.shape[0]}")
print(f"Columns: {final_feature_data.shape[1]}")
print(f"Predictors: {len(COMBINED_FEATURES)}")

Rows: 302
Columns: 61
Predictors: 59


In [35]:
final_feature_data.to_csv(
    FEATURE_ENGINEERED_DATA_FILE,
    index=False
)

print(f"Saved feature-engineered data to:")
print(FEATURE_ENGINEERED_DATA_FILE)

Saved feature-engineered data to:
C:\Users\rebec\Documents\Thesis\Data\Processed\migraine_feature_engineered.csv


In [36]:
print(FEATURE_ENGINEERED_DATA_FILE.exists())

True


In [37]:
print("Feature Engineering Complete")
print("-" * 50)
print(f"Migraine episodes: {len(final_feature_data)}")
print(f"Participants: {final_feature_data[GROUP].nunique()}")
print(f"Candidate predictors: {len(COMBINED_FEATURES)}")
print(f"Total columns saved: {final_feature_data.shape[1]}")
print(f"Target: {TARGET}")
print(f"Grouping variable: {GROUP}")

Feature Engineering Complete
--------------------------------------------------
Migraine episodes: 302
Participants: 48
Candidate predictors: 59
Total columns saved: 61
Target: Severity_VAS
Grouping variable: Registration No.
